## Processor API Endpoints Test

**IMPORTANT: Backend 서버를 사전에 기동해야 합니다.**

```bash
# Terminal에서 실행:
uvicorn src.app.api.main:app --reload
```

Day 3 작업 중 `src/app/api/routers/processors.py`의 모든 엔드포인트 테스트

#### 테스트 대상 엔드포인트
1. `POST /processors/summarize` - 아티클 요약
2. `POST /processors/evaluate` - 중요도 평가
3. `POST /processors/classify` - 카테고리 분류
4. `POST /processors/process` - 전체 파이프라인 처리
5. `POST /processors/batch-process` - 배치 처리
6. `POST /processors/statistics` - 통계 계산

In [1]:
import requests
import json
from pprint import pprint
import time

# API base URL
BASE_URL = "http://127.0.0.1:8000"
PROCESSORS_URL = f"{BASE_URL}/api/processors"

print("✓ Setup complete")

✓ Setup complete


In [2]:
# Server health check
try:
    response = requests.get(f"{BASE_URL}/health", timeout=5.0)
    if response.status_code == 200:
        print("✅ Server is running and healthy!")
        print(f"Response: {response.json()}")
    else:
        print(f"⚠️ Server responded with status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("❌ Cannot connect to server. Please start the backend:")
    print("   uvicorn src.app.api.main:app --reload")
except Exception as e:
    print(f"❌ Health check failed: {e}")

✅ Server is running and healthy!
Response: {'status': 'healthy'}


In [3]:
# 샘플 데이터
sample_paper = {
    "title": "Attention Is All You Need",
    "content": """
    We propose a new simple network architecture, the Transformer,
    based solely on attention mechanisms, dispensing with recurrence
    and convolutions entirely. Experiments on two machine translation
    tasks show these models to be superior in quality while being
    more parallelizable and requiring significantly less time to train.
    Our model achieves 28.4 BLEU on the WMT 2014 English-to-German
    translation task, improving over the existing best results, including
    ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation
    task, our model establishes a new single-model state-of-the-art BLEU
    score of 41.8 after training for 3.5 days on eight GPUs, a small
    fraction of the training costs of the best models from the literature.
    """,
    "url": "https://arxiv.org/abs/1706.03762",
    "source_name": "arXiv",
    "source_type": "paper"
}

sample_news = {
    "title": "OpenAI Announces GPT-4",
    "content": """
    OpenAI today announced GPT-4, the latest milestone in its effort
    to scale up deep learning. GPT-4 is a large multimodal model
    that can accept image and text inputs and produce text outputs.
    While less capable than humans in many real-world scenarios,
    GPT-4 exhibits human-level performance on various professional
    and academic benchmarks.
    """,
    "url": "https://openai.com/research/gpt-4",
    "source_name": "OpenAI",
    "source_type": "news"
}

sample_blog = {
    "title": "BERT: Pre-training of Deep Bidirectional Transformers",
    "content": """
    We introduce a new language representation model called BERT,
    which stands for Bidirectional Encoder Representations from Transformers.
    Unlike recent language representation models, BERT is designed to
    pre-train deep bidirectional representations from unlabeled text
    by jointly conditioning on both left and right context in all layers.
    """,
    "url": "https://arxiv.org/abs/1810.04805",
    "source_name": "arXiv",
    "source_type": "paper"
}

### 1. POST /processors/summarize - 아티클 요약
#### 1.1 한국어 요약 (medium)

In [4]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "language": "ko",
    "length": "medium"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 한국어 요약 성공")
    print(f"\nSummary: {result['summary']}")
    print(f"Language: {result['language']}")
    print(f"Length: {result['length']}")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 한국어 요약 성공

Summary: "Attention Is All You Need" 논문에서는 주의 메커니즘에만 기반한 새로운 네트워크 아키텍처인 트랜스포머를 제안합니다. 이 모델은 순환 및 합성곱을 완전히 배제하고, 두 가지 기계 번역 작업에서 기존 모델보다 더 우수한 품질을 보이며 병렬 처리 가능성과 훈련 시간을 크게 단축합니다. WMT 2014 영어-독일어 번역 작업에서 28.4 BLEU 점수를 기록하며 기존 최고 성과를 2 BLEU 이상 초과했고, 영어-프랑스어 번역 작업에서는 41.8 BLEU로 새로운 단일 모델 최고 성과를 달성했습니다.
Language: ko
Length: medium


#### 1.2 영어 요약 (short)

In [5]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "language": "en",
    "length": "short"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 영어 요약 성공")
    print(f"\nSummary: {result['summary']}")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 영어 요약 성공

Summary: The paper introduces the Transformer, a novel network architecture based entirely on attention mechanisms, eliminating the need for recurrence and convolutions. It demonstrates superior performance in machine translation tasks, achieving a BLEU score of 28.4 for English-to-German and 41.8 for English-to-French on the WMT 2014 dataset, while being more parallelizable and requiring less training time compared to previous models.


#### 1.3 한국어 요약 (long)

In [6]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "language": "ko",
    "length": "long"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 긴 요약 성공")
    print(f"\nSummary length: {len(result['summary'])} chars")
    print(f"Summary: {result['summary'][:200]}...")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 긴 요약 성공

Summary length: 558 chars
Summary: "Attention Is All You Need"는 순환 신경망과 합성곱 신경망을 완전히 배제하고, 주의 메커니즘만을 기반으로 하는 새로운 네트워크 아키텍처인 Transformer를 제안합니다. 이 연구의 배경은 기존의 기계 번역 모델들이 병렬화에 한계가 있고 훈련 시간이 길다는 문제점을 해결하기 위함입니다. Transformer는 두 가지 기계 번역 작업...


### 2. POST /processors/evaluate - 중요도 평가
#### 2.1 기본 평가 (메타데이터 없음)

In [7]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"]
}

response = requests.post(f"{PROCESSORS_URL}/evaluate", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 중요도 평가 성공")
    print(f"\nFinal Score: {result['final_score']:.3f}")
    print(f"Innovation: {result['innovation_score']:.3f}")
    print(f"Relevance: {result['relevance_score']:.3f}")
    print(f"Impact: {result['impact_score']:.3f}")
    print(f"Timeliness: {result['timeliness_score']:.3f}")
    if result.get('reasoning'):
        print(f"\nReasoning: {result['reasoning'][:200]}...")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 중요도 평가 성공

Final Score: 0.850
Innovation: 1.000
Relevance: 1.000
Impact: 1.000
Timeliness: 1.000

Reasoning: The paper 'Attention Is All You Need' introduces the Transformer model, which is a groundbreaking innovation in the field of AI and natural language processing. It replaces traditional recurrent and c...


#### 2.2 메타데이터 포함 평가

In [8]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "metadata": {
        "year": 2017,
        "citations": 50000,
        "venue": "NeurIPS"
    }
}

response = requests.post(f"{PROCESSORS_URL}/evaluate", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 메타데이터 포함 평가 성공")
    print(f"\nFinal Score: {result['final_score']:.3f}")
    print(f"Impact: {result['impact_score']:.3f} (높은 인용 수 반영)")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 메타데이터 포함 평가 성공

Final Score: 0.940
Impact: 1.000 (높은 인용 수 반영)


#### 2.3 뉴스 아티클 평가

In [9]:
payload = {
    "title": sample_news["title"],
    "content": sample_news["content"],
    "metadata": {
        "published_date": "2023-03-14",
        "source": "OpenAI"
    }
}

response = requests.post(f"{PROCESSORS_URL}/evaluate", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 뉴스 평가 성공")
    print(f"\nFinal Score: {result['final_score']:.3f}")
    print(f"Timeliness: {result['timeliness_score']:.3f} (최신성 반영)")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 뉴스 평가 성공

Final Score: 0.780
Timeliness: 0.900 (최신성 반영)


### 3. POST /processors/classify - 카테고리 분류
#### 3.1 논문 분류

In [10]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "source_name": sample_paper["source_name"],
    "url": sample_paper["url"]
}

response = requests.post(f"{PROCESSORS_URL}/classify", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 논문 분류 성공")
    print(f"\nCategory: {result['category']}")
    print(f"Confidence: {result['confidence']:.3f}")
    print(f"Research Field: {result['research_field']}")
    print(f"Sub-fields: {', '.join(result['sub_fields'])}")
    print(f"Keywords: {', '.join(result['keywords'][:10])}")
    if result.get('reasoning'):
        print(f"\nReasoning: {result['reasoning'][:200]}...")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 논문 분류 성공

Category: paper
Confidence: 1.000
Research Field: Machine Learning
Sub-fields: Natural Language Processing, Neural Networks
Keywords: Transformer, attention mechanism, machine translation, BLEU score, parallelizable

Reasoning: The document is a scholarly article published on arXiv, proposing a new network architecture called the Transformer. It focuses on machine translation tasks and presents experimental results, which ar...


#### 3.2 뉴스 분류

In [11]:
payload = {
    "title": sample_news["title"],
    "content": sample_news["content"],
    "source_name": sample_news["source_name"],
    "url": sample_news["url"]
}

response = requests.post(f"{PROCESSORS_URL}/classify", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 뉴스 분류 성공")
    print(f"\nCategory: {result['category']}")
    print(f"Confidence: {result['confidence']:.3f}")
    print(f"Keywords: {', '.join(result['keywords'][:10])}")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 뉴스 분류 성공

Category: news
Confidence: 0.900
Keywords: GPT-4, OpenAI, multimodal model, deep learning, human-level performance


#### 3.3 최소 정보로 분류 (Fallback 테스트)

In [12]:
payload = {
    "title": "Some Random Article",
    "content": "This is a very short article with minimal information.",
    "source_name": "",
    "url": ""
}

response = requests.post(f"{PROCESSORS_URL}/classify", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ Fallback 분류 성공")
    print(f"\nCategory: {result['category']} (likely 'other')")
    print(f"Confidence: {result['confidence']:.3f}")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ Fallback 분류 성공

Category: other (likely 'other')
Confidence: 0.200


### 4. POST /processors/process - 전체 파이프라인 처리
#### 4.1 논문 전체 처리

In [13]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "url": sample_paper["url"],
    "source_name": sample_paper["source_name"],
    "source_type": sample_paper["source_type"],
    "metadata": {
        "year": 2017,
        "citations": 50000
    },
    "summary_length": "medium",
    "summary_language": "ko"
}

start_time = time.time()
response = requests.post(f"{PROCESSORS_URL}/process", json=payload)
elapsed = time.time() - start_time

print(f"Status Code: {response.status_code}")
print(f"Processing Time: {elapsed:.2f}s\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 전체 파이프라인 처리 성공")
    print(f"\nTitle: {result['title']}")
    print(f"Summary: {result['summary'][:150]}...")
    print(f"\nImportance Score: {result['importance_score']:.3f}")
    print(f"  - Innovation: {result['innovation_score']:.3f}")
    print(f"  - Relevance: {result['relevance_score']:.3f}")
    print(f"  - Impact: {result['impact_score']:.3f}")
    print(f"  - Timeliness: {result['timeliness_score']:.3f}")
    print(f"\nCategory: {result['category']}")
    print(f"Research Field: {result['research_field']}")
    print(f"Keywords: {', '.join(result['keywords'][:10])}")
    print(f"\nEmbedding Dimensions: {len(result['embedding'])}")
    print(f"Embedding Sample: {result['embedding'][:5]}")
    print(f"\nProcessed At: {result['processed_at']}")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200
Processing Time: 3.02s

✅ 전체 파이프라인 처리 성공

Title: Attention Is All You Need
Summary: "Attention Is All You Need" 논문에서는 주의 메커니즘만을 기반으로 하는 새로운 네트워크 아키텍처인 Transformer를 제안합니다. 이 모델은 재귀나 합성곱 없이도 작동하며, 기계 번역 작업에서 더 높은 품질을 제공하면서 병렬 처리 가능성이 높고...

Importance Score: 0.940
  - Innovation: 1.000
  - Relevance: 1.000
  - Impact: 1.000
  - Timeliness: 1.000

Category: paper
Research Field: Machine Learning
Keywords: Transformer, attention mechanism, machine translation, BLEU score, parallelizable

Embedding Dimensions: 1536
Embedding Sample: [-0.0010941895889118314, 0.013545675203204155, -0.05201539397239685, -0.01007798220962286, 0.01509891264140606]

Processed At: 2025-12-15T13:35:09.204093


#### 4.2 뉴스 전체 처리 (영어 요약)

In [14]:
payload = {
    "title": sample_news["title"],
    "content": sample_news["content"],
    "url": sample_news["url"],
    "source_name": sample_news["source_name"],
    "source_type": sample_news["source_type"],
    "summary_length": "short",
    "summary_language": "en"
}

start_time = time.time()
response = requests.post(f"{PROCESSORS_URL}/process", json=payload)
elapsed = time.time() - start_time

print(f"Status Code: {response.status_code}")
print(f"Processing Time: {elapsed:.2f}s\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 뉴스 전체 처리 성공")
    print(f"\nSummary (EN): {result['summary']}")
    print(f"Category: {result['category']}")
    print(f"Importance Score: {result['importance_score']:.3f}")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200
Processing Time: 3.44s

✅ 뉴스 전체 처리 성공

Summary (EN): OpenAI announced GPT-4, a large multimodal model capable of processing image and text inputs to generate text outputs. Although it is less capable than humans in many real-world situations, GPT-4 demonstrates human-level performance on several professional and academic benchmarks.
Category: news
Importance Score: 0.766


### 5. POST /processors/batch-process - 배치 처리
#### 5.1 3개 아티클 배치 처리

In [15]:
payload = {
    "articles": [
        {
            "title": sample_paper["title"],
            "content": sample_paper["content"],
            "url": sample_paper["url"],
            "source_name": sample_paper["source_name"],
            "metadata": {"year": 2017, "citations": 50000}
        },
        {
            "title": sample_news["title"],
            "content": sample_news["content"],
            "url": sample_news["url"],
            "source_name": sample_news["source_name"],
            "metadata": {"published_date": "2023-03-14"}
        },
        {
            "title": sample_blog["title"],
            "content": sample_blog["content"],
            "url": sample_blog["url"],
            "source_name": sample_blog["source_name"],
            "metadata": {"year": 2018, "citations": 30000}
        }
    ],
    "max_concurrent": 3,
    "summary_length": "medium",
    "summary_language": "ko"
}

start_time = time.time()
response = requests.post(f"{PROCESSORS_URL}/batch-process", json=payload)
elapsed = time.time() - start_time

print(f"Status Code: {response.status_code}")
print(f"Total Processing Time: {elapsed:.2f}s\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 배치 처리 성공")
    print(f"\nTotal Processed: {result['total']}")
    print(f"Success: {result['success']}")
    print(f"Failed: {result['failed']}")
    success_rate = (result['success'] / result['total'] * 100) if result['total'] > 0 else 0
    print(f"Success Rate: {success_rate:.1f}%")
    print(f"Processing Time: {result['processing_time']:.2f}s")
    
    print(f"\n{'='*80}")
    print("Processed Articles:")
    print(f"{'='*80}")
    
    for i, article in enumerate(result['results'], 1):
        print(f"\n[{i}] {article['title'][:50]}...")
        print(f"    Category: {article['category']}")
        print(f"    Importance: {article['importance_score']:.3f}")
        print(f"    Research Field: {article['research_field']}")
        print(f"    Keywords: {', '.join(article['keywords'][:5])}")
        print(f"    Summary: {article['summary'][:100]}...")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200
Total Processing Time: 5.88s

✅ 배치 처리 성공

Total Processed: 3
Success: 3
Failed: 0
Success Rate: 100.0%
Processing Time: 5.87s

Processed Articles:

[1] Attention Is All You Need...
    Category: paper
    Importance: 0.940
    Research Field: Machine Learning
    Keywords: Transformer, attention mechanism, machine translation, BLEU score, parallelizable
    Summary: "Attention Is All You Need" 논문에서는 Transformer라는 새로운 네트워크 아키텍처를 제안하며, 이는 전적으로 어텐션 메커니즘에 기반하여 순환 및 합성곱...

[2] OpenAI Announces GPT-4...
    Category: news
    Importance: 0.766
    Research Field: Machine Learning
    Keywords: GPT-4, OpenAI, multimodal model, deep learning, human-level performance
    Summary: OpenAI는 최신 딥러닝 확장을 목표로 한 중요한 성과인 GPT-4를 발표했습니다. GPT-4는 이미지와 텍스트 입력을 처리하고 텍스트 출력을 생성할 수 있는 대형 멀티모달 모델...

[3] BERT: Pre-training of Deep Bidirectional Transform...
    Category: paper
    Importance: 0.922
    Research Field: Natural Language Processing
    Keywords: BERT, language representation, tran

### 5.2 배치 처리 (max_concurrent=2)

In [16]:
payload = {
    "articles": [
        {
            "title": sample_paper["title"],
            "content": sample_paper["content"],
            "url": sample_paper["url"],
            "source_name": sample_paper["source_name"]
        },
        {
            "title": sample_news["title"],
            "content": sample_news["content"],
            "url": sample_news["url"],
            "source_name": sample_news["source_name"]
        },
        {
            "title": sample_blog["title"],
            "content": sample_blog["content"],
            "url": sample_blog["url"],
            "source_name": sample_blog["source_name"]
        }
    ],
    "max_concurrent": 2,
    "summary_length": "short",
    "summary_language": "en"
}

start_time = time.time()
response = requests.post(f"{PROCESSORS_URL}/batch-process", json=payload)
elapsed = time.time() - start_time

print(f"Status Code: {response.status_code}")
print(f"Total Processing Time: {elapsed:.2f}s (limited to 2 concurrent)\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 제한된 동시성 배치 처리 성공")
    print(f"\nSuccess: {result['success']}/{result['total']}")
    print(f"Processing Time: {result['processing_time']:.2f}s")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200
Total Processing Time: 6.30s (limited to 2 concurrent)

✅ 제한된 동시성 배치 처리 성공

Success: 3/3
Processing Time: 6.29s


### 6. POST /processors/statistics - 통계 계산
#### 6.1 배치 처리 결과로 통계 계산

In [17]:
# 먼저 배치 처리로 데이터 생성
batch_payload = {
    "articles": [
        {"title": sample_paper["title"], "content": sample_paper["content"], "url": sample_paper["url"], "source_name": sample_paper["source_name"]},
        {"title": sample_news["title"], "content": sample_news["content"], "url": sample_news["url"], "source_name": sample_news["source_name"]},
        {"title": sample_blog["title"], "content": sample_blog["content"], "url": sample_blog["url"], "source_name": sample_blog["source_name"]}
    ],
    "max_concurrent": 3,
    "summary_length": "medium",
    "summary_language": "ko"
}

batch_response = requests.post(f"{PROCESSORS_URL}/batch-process", json=batch_payload)

if batch_response.status_code == 200:
    batch_result = batch_response.json()
    
    # 통계 계산 요청 - 리스트를 직접 전송 (딕셔너리로 감싸지 않음!)
    response = requests.post(f"{PROCESSORS_URL}/statistics", json=batch_result["results"])
    print(f"Status Code: {response.status_code}\n")
    
    if response.status_code == 200:
        result = response.json()
        print(f"✅ 통계 계산 성공")
        print(f"\nTotal Articles: {result['total']}")
        print(f"\nScore Statistics:")
        print(f"  - Average Score: {result['average_score']:.3f}")
        print(f"  - Max Score: {result['max_score']:.3f}")
        print(f"  - Min Score: {result['min_score']:.3f}")
        print(f"  - High Quality Count (≥0.7): {result['high_quality_count']}")
        print(f"\nCategory Distribution:")
        for category, count in result['category_distribution'].items():
            print(f"  - {category}: {count}")
    else:
        print(f"❌ Error: {response.text}")
else:
    print(f"❌ Batch processing failed: {batch_response.text}")

Status Code: 200

✅ 통계 계산 성공

Total Articles: 3

Score Statistics:
  - Average Score: 0.816
  - Max Score: 0.850
  - Min Score: 0.766
  - High Quality Count (≥0.7): 3

Category Distribution:
  - paper: 2
  - news: 1


#### 6.2 빈 리스트 통계 테스트

In [19]:
# 빈 리스트를 직접 전송 (딕셔너리로 감싸지 않음!)
response = requests.post(f"{PROCESSORS_URL}/statistics", json=[])
print(f"Status Code: {response.status_code}\n")

if response.status_code == 200:
    result = response.json()
    print(f"✅ 빈 리스트 통계 처리 성공")
    print(f"\nResult: {result}")
    print(f"Expected: Empty dictionary or zero values")
else:
    print(f"❌ Error: {response.text}")

Status Code: 200

✅ 빈 리스트 통계 처리 성공

Result: {'total': 0, 'category_distribution': {}, 'average_score': 0.0, 'max_score': 0.0, 'min_score': 0.0, 'high_quality_count': 0}
Expected: Empty dictionary or zero values


### 7. 에러 처리 테스트
#### 7.1 필수 필드 누락 (title 없음)

In [20]:
payload = {
    "content": sample_paper["content"],
    "language": "ko",
    "length": "medium"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 422:
    print(f"✅ 예상된 유효성 검사 오류 발생")
    print(f"Error Detail: {response.json()}")
else:
    print(f"❌ Unexpected response: {response.text}")

Status Code: 422

✅ 예상된 유효성 검사 오류 발생
Error Detail: {'detail': [{'type': 'missing', 'loc': ['body', 'title'], 'msg': 'Field required', 'input': {'content': '\n    We propose a new simple network architecture, the Transformer,\n    based solely on attention mechanisms, dispensing with recurrence\n    and convolutions entirely. Experiments on two machine translation\n    tasks show these models to be superior in quality while being\n    more parallelizable and requiring significantly less time to train.\n    Our model achieves 28.4 BLEU on the WMT 2014 English-to-German\n    translation task, improving over the existing best results, including\n    ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation\n    task, our model establishes a new single-model state-of-the-art BLEU\n    score of 41.8 after training for 3.5 days on eight GPUs, a small\n    fraction of the training costs of the best models from the literature.\n    ', 'language': 'ko', 'length': 'medium'}}]}


#### 7.2 잘못된 language 값

In [21]:
payload = {
    "title": sample_paper["title"],
    "content": sample_paper["content"],
    "language": "invalid_language",
    "length": "medium"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code == 422:
    print(f"✅ 예상된 유효성 검사 오류 발생 (잘못된 language)")
    error_detail = response.json()
    print(f"Error Detail: {error_detail}")
else:
    print(f"❌ Unexpected response: {response.text}")

Status Code: 200

❌ Unexpected response: {"summary":"The paper \"Attention Is All You Need\" introduces the Transformer, a novel network architecture that relies entirely on attention mechanisms, eliminating the need for recurrence and convolutions. This architecture demonstrates superior performance in machine translation tasks, achieving a BLEU score of 28.4 on the WMT 2014 English-to-German task and setting a new single-model state-of-the-art BLEU score of 41.8 on the English-to-French task. The Transformer is more parallelizable and significantly reduces training time, requiring only 3.5 days on eight GPUs, which is considerably less than previous models. These results highlight the efficiency and effectiveness of the Transformer model in comparison to traditional approaches.","language":"invalid_language","length":"medium"}


#### 7.3 빈 content 처리

In [22]:
payload = {
    "title": "Empty Article",
    "content": "",
    "language": "ko",
    "length": "medium"
}

response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
print(f"Status Code: {response.status_code}\n")

if response.status_code in [400, 422, 500]:
    print(f"✅ 빈 content에 대한 오류 처리 확인")
    print(f"Error: {response.json()}")
elif response.status_code == 200:
    print(f"⚠️ 빈 content가 처리됨 (의도된 동작인지 확인 필요)")
    print(f"Result: {response.json()}")
else:
    print(f"❌ Unexpected response: {response.text}")

Status Code: 200

⚠️ 빈 content가 처리됨 (의도된 동작인지 확인 필요)
Result: {'summary': '이 문서는 "Empty Article"이라는 제목을 가지고 있으나, 실제로는 내용이 제공되지 않았습니다. 따라서 이 문서에서 추출할 수 있는 핵심 아이디어나 주요 발견은 없습니다. 내용이 비어 있어 추가적인 정보나 분석이 불가능한 상태입니다.', 'language': 'ko', 'length': 'medium'}


### 8. 성능 벤치마크
#### 8.1 요약 성능 비교 (short vs medium vs long)

In [23]:
print("요약 길이별 성능 벤치마크\n" + "="*80)

lengths = ["short", "medium", "long"]
results = {}

for length in lengths:
    payload = {
        "title": sample_paper["title"],
        "content": sample_paper["content"],
        "language": "ko",
        "length": length
    }
    
    start_time = time.time()
    response = requests.post(f"{PROCESSORS_URL}/summarize", json=payload)
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        result = response.json()
        summary_length = len(result['summary'])
        results[length] = {"time": elapsed, "chars": summary_length}
        print(f"\n{length.upper():8s}: {elapsed:.2f}s | {summary_length:4d} chars")
    else:
        print(f"\n{length.upper():8s}: Failed - {response.text}")

print("\n" + "="*80)
print("✅ 요약 성능 벤치마크 완료")

요약 길이별 성능 벤치마크

SHORT   : 1.86s |  226 chars

MEDIUM  : 1.74s |  310 chars

LONG    : 2.72s |  524 chars

✅ 요약 성능 벤치마크 완료


#### 8.2 배치 처리 성능 (동시성 비교)

In [24]:
print("배치 처리 동시성 성능 벤치마크\n" + "="*80)

articles = [
    {"title": sample_paper["title"], "content": sample_paper["content"], "url": sample_paper["url"], "source_name": sample_paper["source_name"]},
    {"title": sample_news["title"], "content": sample_news["content"], "url": sample_news["url"], "source_name": sample_news["source_name"]},
    {"title": sample_blog["title"], "content": sample_blog["content"], "url": sample_blog["url"], "source_name": sample_blog["source_name"]}
]

concurrency_levels = [1, 2, 3]
benchmark_results = {}

for max_concurrent in concurrency_levels:
    payload = {
        "articles": articles,
        "max_concurrent": max_concurrent,
        "summary_length": "short",
        "summary_language": "ko"
    }
    
    start_time = time.time()
    response = requests.post(f"{PROCESSORS_URL}/batch-process", json=payload)
    elapsed = time.time() - start_time
    
    if response.status_code == 200:
        result = response.json()
        benchmark_results[max_concurrent] = elapsed
        print(f"\nConcurrency {max_concurrent}: {elapsed:.2f}s | Success: {result['success']}/{result['total']}")
    else:
        print(f"\nConcurrency {max_concurrent}: Failed - {response.text}")

print("\n" + "="*80)
if len(benchmark_results) > 1:
    speedup_2 = benchmark_results[1] / benchmark_results[2] if 2 in benchmark_results else 0
    speedup_3 = benchmark_results[1] / benchmark_results[3] if 3 in benchmark_results else 0
    print(f"Speedup (2 vs 1): {speedup_2:.2f}x")
    print(f"Speedup (3 vs 1): {speedup_3:.2f}x")
print("✅ 배치 처리 성능 벤치마크 완료")

배치 처리 동시성 성능 벤치마크

Concurrency 1: 10.83s | Success: 3/3

Concurrency 2: 8.95s | Success: 3/3

Concurrency 3: 4.42s | Success: 3/3

Speedup (2 vs 1): 1.21x
Speedup (3 vs 1): 2.45x
✅ 배치 처리 성능 벤치마크 완료


### 9. End-to-End 워크플로우 테스트
#### 9.1 전체 워크플로우: 요약 → 평가 → 분류 → 통합 처리

In [25]:
print("End-to-End 워크플로우 테스트\n" + "="*80)

test_article = sample_paper.copy()

# Step 1: 요약
print("\n[Step 1] 요약 생성...")
summary_payload = {
    "title": test_article["title"],
    "content": test_article["content"],
    "language": "ko",
    "length": "medium"
}
summary_response = requests.post(f"{PROCESSORS_URL}/summarize", json=summary_payload)
if summary_response.status_code == 200:
    summary = summary_response.json()["summary"]
    print(f"✅ 요약: {summary[:100]}...")
else:
    print(f"❌ 요약 실패")
    summary = None

# Step 2: 평가
print("\n[Step 2] 중요도 평가...")
eval_payload = {
    "title": test_article["title"],
    "content": test_article["content"],
    "metadata": {"year": 2017, "citations": 50000}
}
eval_response = requests.post(f"{PROCESSORS_URL}/evaluate", json=eval_payload)
if eval_response.status_code == 200:
    eval_result = eval_response.json()
    print(f"✅ 중요도 점수: {eval_result['final_score']:.3f}")
else:
    print(f"❌ 평가 실패")
    eval_result = None

# Step 3: 분류
print("\n[Step 3] 카테고리 분류...")
classify_payload = {
    "title": test_article["title"],
    "content": test_article["content"],
    "source_name": test_article["source_name"],
    "url": test_article["url"]
}
classify_response = requests.post(f"{PROCESSORS_URL}/classify", json=classify_payload)
if classify_response.status_code == 200:
    classify_result = classify_response.json()
    print(f"✅ 카테고리: {classify_result['category']} ({classify_result['research_field']})")
else:
    print(f"❌ 분류 실패")
    classify_result = None

# Step 4: 통합 처리 (모든 단계 한번에)
print("\n[Step 4] 통합 파이프라인 처리...")
process_payload = {
    "title": test_article["title"],
    "content": test_article["content"],
    "url": test_article["url"],
    "source_name": test_article["source_name"],
    "source_type": test_article["source_type"],
    "metadata": {"year": 2017, "citations": 50000},
    "summary_length": "medium",
    "summary_language": "ko"
}
process_response = requests.post(f"{PROCESSORS_URL}/process", json=process_payload)
if process_response.status_code == 200:
    process_result = process_response.json()
    print(f"✅ 통합 처리 완료")
    print(f"   - Summary: {process_result['summary'][:100]}...")
    print(f"   - Score: {process_result['importance_score']:.3f}")
    print(f"   - Category: {process_result['category']}")
    print(f"   - Embedding: {len(process_result['embedding'])} dims")
else:
    print(f"❌ 통합 처리 실패")

print("\n" + "="*80)
print("✅ End-to-End 워크플로우 테스트 완료")

End-to-End 워크플로우 테스트

[Step 1] 요약 생성...
✅ 요약: "Attention Is All You Need" 논문에서는 주의 메커니즘만을 기반으로 하는 새로운 네트워크 아키텍처인 Transformer를 제안합니다. 이 모델은 반복과 합성을...

[Step 2] 중요도 평가...
✅ 중요도 점수: 0.940

[Step 3] 카테고리 분류...
✅ 카테고리: paper (Machine Learning)

[Step 4] 통합 파이프라인 처리...
✅ 통합 처리 완료
   - Summary: "Attention Is All You Need" 논문은 완전히 어텐션 메커니즘에 기반한 새로운 네트워크 아키텍처인 트랜스포머를 제안합니다. 이 모델은 순환과 합성곱을 사용하지 않...
   - Score: 0.940
   - Category: paper
   - Embedding: 1536 dims

✅ End-to-End 워크플로우 테스트 완료


### 10. 테스트 요약
#### 전체 테스트 결과

In [26]:
print("\n" + "="*80)
print("✅ Processor API 테스트 완료")
print("="*80)
print("""
테스트 완료된 엔드포인트:
  1. POST /processors/summarize (한국어/영어, short/medium/long)
  2. POST /processors/evaluate (메타데이터 포함/미포함)
  3. POST /processors/classify (논문/뉴스/블로그)
  4. POST /processors/process (전체 파이프라인)
  5. POST /processors/batch-process (배치 처리)
  6. POST /processors/statistics (통계 계산)
  7. 에러 처리 (필수 필드 누락, 잘못된 값)
  8. 성능 벤치마크 (요약 길이별, 동시성별)
  9. End-to-End 워크플로우

모든 테스트가 정상적으로 완료되었습니다! 🎉
""")
print("="*80)


✅ Processor API 테스트 완료

테스트 완료된 엔드포인트:
  1. POST /processors/summarize (한국어/영어, short/medium/long)
  2. POST /processors/evaluate (메타데이터 포함/미포함)
  3. POST /processors/classify (논문/뉴스/블로그)
  4. POST /processors/process (전체 파이프라인)
  5. POST /processors/batch-process (배치 처리)
  6. POST /processors/statistics (통계 계산)
  7. 에러 처리 (필수 필드 누락, 잘못된 값)
  8. 성능 벤치마크 (요약 길이별, 동시성별)
  9. End-to-End 워크플로우

모든 테스트가 정상적으로 완료되었습니다! 🎉

